# Rule-Consistency Auditor
- Francesco Buda, francesco.buda3@studio.unibo.it
- Emanuele Sanchi, emanuele.sanchi@studio.unibo.it
- Tommaso Severi, tommaso.severi2@studio.unibo.it

## Import libraries

In [1]:
import time
import carla_utils
import data_list_bind
import carla
import datetime
import json
import threading
from rca_display import RCADisplay
from controller.state_machine import StateMachine
import parsing_utils
from models.state import StateType

pygame 2.6.1 (SDL 2.28.4, Python 3.7.12)
Hello from the pygame community. https://www.pygame.org/contribute.html


## Setup log file and CARLA world

In [2]:
log_filename = f"logs/rca_log_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
world, spectator, client = carla_utils.world_connect()

## Spawn vehicles

In [3]:
AUTOPILOT_VEHICLE_COUNT = 10
for i in range(AUTOPILOT_VEHICLE_COUNT):
    carla_utils.spawn_random_vehicle_no_bike(world, spawn_index=i, autopilot=True)
ego_vehicle = carla_utils.spawn_vehicle(world, spawn_index=10, autopilot=False)

## Setup binder thread

In [ ]:
binder = data_list_bind.VariableBinder(world, ego_vehicle)
scene_data_shared = {}
scene_data_lock = threading.Lock()
binder_stop_event = threading.Event()

def binder_thread_fn():
    while not binder_stop_event.is_set():
        data = binder.compute_scene_data()
        with scene_data_lock:
            scene_data_shared.update(data)
        time.sleep(0.05)

binder_thread = threading.Thread(target=binder_thread_fn, daemon=True)
binder_thread.start()

: 

## Main control loop

In [ ]:
state_machines = [StateMachine(rule) for rule in parsing_utils.parse_rule_files("admin/rules/")]

display = RCADisplay(world, ego_vehicle)
display.start()

frame_count = 0
log_buffer  = []
FLUSH_EVERY = 20

tick_event = threading.Event()

def on_server_tick(snapshot):
    tick_event.set()

world.on_tick(on_server_tick)

try:
    while display.running:
        tick_event.wait(timeout=0.1)
        tick_event.clear()

        if not display.tick():
            break

        ego_vehicle.apply_control(display.get_control())

        with scene_data_lock:
            scene_data = dict(scene_data_shared)
        timestamp = datetime.datetime.now().strftime('%H:%M:%S.%f')[:-3]

        log_entry = {
            'frame':     frame_count,
            'timestamp': timestamp,
            'reverse':   display.reverse,
            'control': {
                'throttle': round(float(display.control.throttle), 3),
                'brake':    round(float(display.control.brake),    3),
                'steer':    round(float(display.control.steer),    3),
            },
            'scene_data': {
                k: str(v) if hasattr(v, '__dict__') else v
                for k, v in scene_data.items()
            }
        }
        log_buffer.append(log_entry)

        if len(log_buffer) >= FLUSH_EVERY:
            with open(log_filename, 'a') as f:
                for entry in log_buffer:
                    f.write(json.dumps(entry) + '\n')
            log_buffer.clear()

        if frame_count % 10 == 0:
            for sm in state_machines:
                current_state = sm.evaluate(scene_data)
                if current_state.type == StateType.VIOLATION:
                    print(f"Rule violation detected: {sm._rule.name} at frame {frame_count}")

        frame_count += 1

finally:
    binder_stop_event.set()
    binder_thread.join(timeout=2.0)

    if log_buffer:
        with open(log_filename, 'a') as f:
            for entry in log_buffer:
                f.write(json.dumps(entry) + '\n')

    display.destroy()

    try:
        ego_vehicle.destroy()
    except Exception:
        pass
    try:
        for actor in world.get_actors().filter('vehicle.*'):
            actor.destroy()
    except Exception:
        pass

    print(f"Simulazione terminata dopo {frame_count} frame. Log: {log_filename}")

## Cleanup

In [ ]:
print("Destroying all vehicles in the simulator...")
for actor in world.get_actors().filter('vehicle.*'):
    try:
        actor.destroy()
        print(f"  Destroyed: {actor.id}")
    except Exception as e:
        print(f"  Skip {actor.id}: {e}")
print("All vehicles cleaned up. Ready for next run!")